# Week 7: LLMs through an API

A large language model (LLM) predicts likely next words. Today you'll call one from Python, steer it with prompts, and get it to pull structured data out of a Nigerian bank SMS.

**Before you start:** add your API key as a secret. On Kaggle: **Add-ons > Secrets**, name it `GROQ_API_KEY` (or `GEMINI_API_KEY`), and switch it on for this notebook. Turn **Internet on** in the settings panel. On Colab: click the key icon on the left and add the same name.

Run each cell with **Shift + Enter**. Every exercise has a check cell that prints **PASS** or a hint.
At the end you get a completion code for the submission form.

In [ ]:
# Run this cell first. You don't need to read it: it holds the self-checks.
EMAIL = ""  # <- the email you applied with

import hashlib, json, os, re

_passed = set()

def ok(n):
    _passed.add(n)
    print(f"PASS: exercise {n} done.")

def hint(message):
    print(f"Not yet. Hint: {message}")

def free_text(n, text, min_sentences):
    parts = [p for p in re.split(r"[.!?]+", text or "") if len(p.split()) >= 3]
    if len(parts) >= min_sentences:
        print("Saved. A TA will read this answer.")
        ok(n)
    else:
        hint(f"write at least {min_sentences} full sentences inside the quotes.")

def completion_code(week, exercises):
    if "@" not in EMAIL:
        return print("Put your email in the first cell, run it, then run this cell again.")
    missing = sorted(set(exercises) - _passed)
    if missing:
        return print(f"Almost there. Not passed yet: {missing}. Run those exercises and checks again.")
    digest = hashlib.sha256(f"akada-labs-cohort-1|{week}|{EMAIL.strip().lower()}".encode()).hexdigest()
    print(f"Well done! Your completion code is: AK-W{week:02d}-{digest[:6].upper()}")
    print("Paste it into the lab submission form.")

def get_secret(name):
    """Read an API key from Kaggle or Colab secrets. Never paste keys into a cell."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    raise RuntimeError(
        f"No secret called {name}. On Kaggle: Add-ons > Secrets. On Colab: the key icon on the left. "
        "Add it there, switch it on for this notebook, then run this cell again.")

print("Ready.")

In [ ]:
!pip install -q openai

## Exercise 1: Your first call

Run the provider cell (it reads your key from secrets), then ask the model a question and store its text in `reply`.

```python
response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "..."}])
reply = response.choices[0].message.content
```

In [ ]:
# Pick one provider. Both have a free tier.
PROVIDER = "groq"   # or "gemini"

PROVIDERS = {
    # name: (secret name, OpenAI-compatible endpoint, model)
    "groq":   ("GROQ_API_KEY",   "https://api.groq.com/openai/v1", "llama-3.1-8b-instant"),
    "gemini": ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.5-flash"),
}
# Model names change. If you get a "model not found" error, check the provider's model list
# (console.groq.com/docs/models or ai.google.dev/gemini-api/docs/models) and update MODEL.

from openai import OpenAI

SECRET_NAME, BASE_URL, MODEL = PROVIDERS[PROVIDER]
client = OpenAI(api_key=get_secret(SECRET_NAME), base_url=BASE_URL)
print("Using", PROVIDER, MODEL)

reply = None

print(reply)

In [ ]:
try:
    if isinstance(reply, str) and reply.strip():
        ok(1)
    else:
        hint("reply should be response.choices[0].message.content")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

A helper so you don't repeat yourself:

In [ ]:
def ask(prompt, system="You are a helpful assistant.", temperature=0.7):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

print(ask("Say hello in Nigerian Pidgin, in one line."))

## Exercise 2: System prompt and temperature

Ask the same question twice with different settings. For `calm`, use a plain system prompt and `temperature=0`. For `wild`, give it a personality ("You are a Lagos danfo conductor...") and `temperature=1.2`. Compare them.

In [ ]:
question = "Explain what inflation is in two sentences."
calm = None
wild = None

print(calm, "\n---\n", wild)

In [ ]:
try:
    if not (isinstance(calm, str) and isinstance(wild, str)) or calm.strip() == wild.strip():
        hint("calm and wild should both come from ask(...), with different system prompts and temperatures.")
    else:
        ok(2)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Structured output

LLMs return text, but programs need data. Ask for **JSON** and say exactly which fields you want.
LLMs sometimes wrap JSON in ```` ``` ```` fences, so `parse_json` strips them.

In [ ]:
def parse_json(text):
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    return json.loads(text)

sms = "Acct:****4521 Amt:NGN15,500.00 CR Desc:TRF FROM ADEBAYO OLUWASEUN/OPAY Date:12-Mar-2026 14:22 Bal:NGN48,210.35"

## Exercise 3: Parse a bank alert

Write a prompt that turns `sms` into JSON with exactly 3 fields: `name` (who sent the money), `amount` (a number) and `date` (YYYY-MM-DD). Use `temperature=0`. Store the parsed result in `data`.

In [ ]:
prompt = f"""

{sms}
"""

data = None

print(data)

In [ ]:
try:
    missing = {"name", "amount", "date"} - set(data or {})
    if missing:
        hint(f"the JSON is missing {sorted(missing)}. Name each field in your prompt.")
    elif abs(float(str(data["amount"]).replace(",", "")) - 15500) > 0.01:
        hint("amount should be 15500, as a number without the currency.")
    elif str(data["date"]) != "2026-03-12":
        hint("date should be 2026-03-12. Show the format you want in the prompt.")
    else:
        ok(3)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 4: Handle failure

APIs fail: wrong keys, rate limits, no internet. Write `safe_ask(prompt, client)` that returns the reply, or a short message starting with `"Error:"` if anything goes wrong. It must never crash.

The check calls it with a client that has a bad key.

In [ ]:
def safe_ask(prompt, client):
    # use try / except around the API call
    pass

In [ ]:
try:
    bad_client = OpenAI(api_key="not-a-real-key", base_url=BASE_URL, max_retries=0)
    result = safe_ask("hello", bad_client)
    if isinstance(result, str) and result.startswith("Error:"):
        print("Your function returned:", result[:120])
        ok(4)
    else:
        hint("catch the exception with except Exception as e: and return f\"Error: {e}\".")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

**Keep your key safe.** Never paste it into a cell, a GitHub repo or a WhatsApp group. If you ever do, delete it in the provider's console and make a new one.

## Finish

Run the cell below. If every exercise passed, it prints your completion code.

In [ ]:
completion_code(7, [1, 2, 3, 4])